# 05 — Old vs New Document Comparison (Phase 5)

**Project:** DGCA Agentic AI Regulatory Intelligence & Monitoring System
**Phase:** 5 of 8 — Revision Detection & AI-Assisted Comparison

## 1. Project Overview

When DGCA republishes a revised regulation, the useful question isn't "what
does this document say" (Phase 4 already extracted that) but **"what
changed since the last version"**. This notebook adds that: given a new
publication, try to find its previous version among what's already in the
Phase 3 database, diff the two texts at a meaningful granularity, and
produce a structured comparison — added / removed / modified / unchanged —
that deliberately ignores PDF-extraction formatting noise (line wraps,
irregular spacing) so it never gets reported as a "regulatory change".

```
NEW publication (from Phase 3 DB)
        |
        v
find_previous_version()  -- same category, title-similarity match after
        |                    stripping "Revision N"/"Amendment N" style suffixes
        v
   OLD text (Phase 4 .txt)  +  NEW text (Phase 4 .txt)
        |
        v
normalize_for_diff()  -- collapse all whitespace BEFORE diffing, so line-wrap
        |                 differences never even become diff hunks
        v
diff_documents()  -- sentence-level difflib.SequenceMatcher -> added/removed/modified
        |
        v
AI Comparison Agent:
   OPENAI_API_KEY configured -> LLM turns the diff into the structured JSON below
   not configured             -> deterministic rule-based fallback (same schema),
                                  clearly logged as a fallback, not pretended as AI
        |
        v
{
  "overall_change_level": "HIGH",
  "total_changes": 13,
  "added": [], "removed": [], "modified": [],
  "key_changes": [],
  "effective_date": "Not specified in the publication.",
  "operational_impact": "...",
  "impacted_departments": [],
  "recommended_actions": []
}
```

**Ground rule enforced throughout:** never invent a regulatory requirement
that isn't actually present in the diff; when information genuinely isn't
available, the literal string `"Not specified in the publication."` is used
-- both in the fallback and as an explicit instruction to the LLM.

**Scope of this phase:** comparing two already-extracted texts. Turning the
comparison into an executive summary/impact email is Phase 6.


## 2. Imports

In [ ]:
import difflib
import json
import logging
import os
import re
from psycopg import Connection
from psycopg.rows import DictRow
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Optional, Tuple

from dotenv import load_dotenv
from pydantic import BaseModel, Field

try:
    from openai import OpenAI
    _OPENAI_SDK_AVAILABLE = True
except ImportError:
    _OPENAI_SDK_AVAILABLE = False

print("Imports OK")


## 3. Configuration

Reuses the same `OPENAI_API_KEY`/`LLM_MODEL` variables as Phase 1, loaded
from the same `config/.env`. Nothing new to configure for this phase; if
`OPENAI_API_KEY` isn't set, the notebook still runs completely -- it just
exercises the deterministic fallback path instead of a real LLM call, same
pattern as Phase 1's AI summary.


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ENV_PATH = PROJECT_ROOT / "config" / ".env"
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
COMPARISONS_DIR = PROJECT_ROOT / "data" / "processed" / "comparisons"
LOG_DIR = PROJECT_ROOT / "logs"

COMPARISONS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(dotenv_path=ENV_PATH)

_PLACEHOLDER_PATTERNS = ("your-", "00000000-0000-0000-0000-000000000000", "sk-your-")


def _is_placeholder(value: Optional[str]) -> bool:
    if not value:
        return True
    return any(value.startswith(p) for p in _PLACEHOLDER_PATTERNS)


@dataclass
class ComparisonConfig:
    openai_api_key: Optional[str] = None
    llm_model: str = "gpt-4o-mini"

    def openai_configured(self) -> bool:
        return not _is_placeholder(self.openai_api_key)


def load_config() -> ComparisonConfig:
    return ComparisonConfig(
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        llm_model=os.getenv("LLM_MODEL", "gpt-4o-mini"),
    )


config = load_config()
print(f"OPENAI_API_KEY configured: {config.openai_configured()}")
print(f"LLM_MODEL: {config.llm_model}")

from pipeline.storage import get_connection


## 4. Logging

In [ ]:
LOG_FILE = LOG_DIR / f"comparison_{datetime.now(timezone.utc).strftime('%Y%m%d')}.log"

logger = logging.getLogger("dgca_comparison")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_formatter = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.propagate = False

logger.info("Comparison logger initialized.")
logger.info(f"Log file: {LOG_FILE}")


## 5. Data Model

`ComparisonResult` matches the project brief's JSON schema exactly.


In [ ]:
class ComparisonResult(BaseModel):
    overall_change_level: str = "NONE"  # NONE | LOW | MEDIUM | HIGH
    total_changes: int = 0
    added: List[str] = Field(default_factory=list)
    removed: List[str] = Field(default_factory=list)
    modified: List[str] = Field(default_factory=list)
    key_changes: List[str] = Field(default_factory=list)
    effective_date: str = "Not specified in the publication."
    operational_impact: str = "Not specified in the publication."
    impacted_departments: List[str] = Field(default_factory=list)
    recommended_actions: List[str] = Field(default_factory=list)


print("ComparisonResult model ready.")


## 6. Text Normalization & Sentence Splitting

PDF-extracted text (Phase 4's `.txt` output) has irregular line breaks from
the original page layout, not clean paragraphs. Collapsing **all**
whitespace to single spaces *before* diffing means two versions that only
differ in line-wrapping become textually identical and produce zero diff
hunks -- rather than trying to filter out "formatting-looking" hunks after
the fact, which is much less reliable.


In [ ]:
def normalize_for_diff(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.;:])\s+")


def split_into_units(text: str, min_len: int = 8) -> List[str]:
    normalized = normalize_for_diff(text)
    raw_units = _SENTENCE_SPLIT_RE.split(normalized)
    return [u.strip() for u in raw_units if len(u.strip()) >= min_len]


print("Normalization functions ready.")


## 7. Diff Engine

Sentence-level `difflib.SequenceMatcher`. A `replace` block (old sentences
swapped for new ones in the same position) is paired up index-wise as
"modified"; any length mismatch within a replace block spills over into
added/removed, since that's genuinely new or genuinely deleted content.


In [ ]:
@dataclass
class DiffResult:
    added: List[str] = field(default_factory=list)
    removed: List[str] = field(default_factory=list)
    modified: List[Tuple[str, str]] = field(default_factory=list)
    unchanged_count: int = 0


def diff_documents(old_text: str, new_text: str) -> DiffResult:
    old_units = split_into_units(old_text)
    new_units = split_into_units(new_text)

    matcher = difflib.SequenceMatcher(None, old_units, new_units, autojunk=False)
    result = DiffResult()

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            result.unchanged_count += (i2 - i1)
        elif tag == "insert":
            result.added.extend(new_units[j1:j2])
        elif tag == "delete":
            result.removed.extend(old_units[i1:i2])
        elif tag == "replace":
            old_chunk = old_units[i1:i2]
            new_chunk = new_units[j1:j2]
            paired = min(len(old_chunk), len(new_chunk))
            for k in range(paired):
                result.modified.append((old_chunk[k], new_chunk[k]))
            if len(old_chunk) > paired:
                result.removed.extend(old_chunk[paired:])
            if len(new_chunk) > paired:
                result.added.extend(new_chunk[paired:])

    return result


print("Diff engine ready.")


## 8. Previous-Version Identification

Matches within the same `category`, after stripping revision-style suffixes
("Revision 2", "Amendment 3", "Issue 4") from both titles so "CAR Section 1
— Revision 1" and "CAR Section 1 — Revision 2" are recognized as the same
underlying document. Read-only against Phase 3's database -- this notebook
never writes to `publications`.


In [ ]:
_REVISION_SUFFIX_RE = re.compile(r"\b(revision|rev\.?|amendment|issue)\s*\d+\b", re.IGNORECASE)


def normalize_title_for_matching(title: str) -> str:
    stripped = _REVISION_SUFFIX_RE.sub("", title)
    return re.sub(r"\s+", " ", stripped).strip().lower()


def titles_likely_same_document(title_a: str, title_b: str, threshold: float = 0.85) -> bool:
    norm_a = normalize_title_for_matching(title_a)
    norm_b = normalize_title_for_matching(title_b)
    if not norm_a or not norm_b:
        return False
    ratio = difflib.SequenceMatcher(None, norm_a, norm_b).ratio()
    return ratio >= threshold


def find_previous_version(conn: Connection, publication_id: str) -> Optional[DictRow]:
    current = conn.execute(
        "SELECT * FROM publications WHERE publication_id = %s", (publication_id,)
    ).fetchone()
    if current is None:
        return None

    candidates = conn.execute(
        "SELECT * FROM publications WHERE category = %s AND publication_id != %s ORDER BY first_seen_at",
        (current["category"], publication_id),
    ).fetchall()

    for candidate in candidates:
        if titles_likely_same_document(current["title"], candidate["title"]):
            return candidate
    return None


print("Previous-version identification ready.")


## 9. AI Comparison Agent

Same honest pattern as Phase 1's AI summary: if `OPENAI_API_KEY` is
configured, the LLM turns the pre-computed diff into the structured JSON
result, explicitly instructed to prioritize substance over formatting and to
never invent requirements. If not configured (the case in this environment),
a deterministic rule-based builder produces the identical schema directly
from the diff -- logged clearly as a fallback, never silently pretended to
be AI-generated.


In [ ]:
def _fallback_comparison(diff: DiffResult) -> ComparisonResult:
    total = len(diff.added) + len(diff.removed) + len(diff.modified)
    if total == 0:
        level = "NONE"
    elif total <= 2:
        level = "LOW"
    elif total <= 6:
        level = "MEDIUM"
    else:
        level = "HIGH"

    key_changes: List[str] = []
    for item in diff.added[:5]:
        key_changes.append(f"Added: {item}")
    for item in diff.removed[:5]:
        key_changes.append(f"Removed: {item}")
    for old, new in diff.modified[:5]:
        key_changes.append(f"Modified: \'{old}\' -> \'{new}\'")

    operational_impact = (
        "Not specified in the publication." if total == 0
        else f"{total} textual change(s) detected between the two versions; manual review recommended to assess operational impact."
    )
    recommended_actions = [] if total == 0 else ["Review the highlighted changes above against current procedures."]

    return ComparisonResult(
        overall_change_level=level,
        total_changes=total,
        added=list(diff.added),
        removed=list(diff.removed),
        modified=[f"{o} -> {n}" for o, n in diff.modified],
        key_changes=key_changes,
        effective_date="Not specified in the publication.",
        operational_impact=operational_impact,
        impacted_departments=[],
        recommended_actions=recommended_actions,
    )


def generate_ai_comparison(diff: DiffResult, config: ComparisonConfig) -> ComparisonResult:
    if not (_OPENAI_SDK_AVAILABLE and config.openai_configured()):
        logger.warning("OPENAI_API_KEY not configured -- using deterministic fallback comparison.")
        return _fallback_comparison(diff)

    diff_summary = json.dumps(
        {
            "added": diff.added[:30],
            "removed": diff.removed[:30],
            "modified": [{"old": o, "new": n} for o, n in diff.modified[:30]],
        },
        indent=2,
    )
    prompt = (
        "You are a regulatory-change analyst comparing two versions of a DGCA publication.\n"
        "Below is a pre-computed textual diff (added/removed/modified sentences) between the OLD\n"
        "and NEW versions. Using ONLY this diff, produce a JSON object with EXACTLY these keys:\n"
        "overall_change_level (NONE|LOW|MEDIUM|HIGH), total_changes (integer), added (list of strings),\n"
        "removed (list of strings), modified (list of strings), key_changes (list of strings),\n"
        "effective_date (string), operational_impact (string), impacted_departments (list of strings),\n"
        "recommended_actions (list of strings).\n\n"
        "Rules:\n"
        "- Prioritize substantive regulatory/operational changes over formatting differences.\n"
        "- Never invent requirements not present in the diff.\n"
        "- If information is not available, use exactly: \"Not specified in the publication.\"\n\n"
        f"DIFF:\n{diff_summary}\n"
    )
    try:
        client = OpenAI(api_key=config.openai_api_key)
        response = client.chat.completions.create(
            model=config.llm_model,
            messages=[
                {"role": "system", "content": "You produce accurate, structured JSON regulatory change analyses. Output valid JSON only."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
            response_format={"type": "json_object"},
        )
        parsed = json.loads(response.choices[0].message.content)
        result = ComparisonResult(**parsed)
        logger.info(f"AI comparison generated via OpenAI model \'{config.llm_model}\'.")
        return result
    except Exception as exc:
        logger.error(f"AI comparison generation failed ({exc.__class__.__name__}). Falling back to deterministic comparison.")
        return _fallback_comparison(diff)


print("AI comparison (+ fallback) ready.")


## 10. Orchestrator — `compare_documents()`

In [ ]:
def compare_documents(old_text: str, new_text: str, config: ComparisonConfig) -> Tuple[DiffResult, ComparisonResult]:
    diff = diff_documents(old_text, new_text)
    logger.info(
        f"Diff computed: {len(diff.added)} added, {len(diff.removed)} removed, "
        f"{len(diff.modified)} modified, {diff.unchanged_count} unchanged units."
    )
    result = generate_ai_comparison(diff, config)
    return diff, result


print("compare_documents() ready.")


## 11. Test Data — real DGCA text + constructed OLD/NEW variants

DGCA's "What's New" listing shows only the current state of each
publication, not full revision history, so our real scraped data has no
genuine old/new pair to compare (confirmed: Phase 3's database has no two
publications that are actually revisions of each other). To test the diff
engine meaningfully, `BASE_TEXT` is a real slice of Phase 4's extracted
regulation text, and each test below constructs a specific, deliberate
variant of it -- added / removed / modified / formatting-only -- so each
diff category can be verified independently and precisely.


In [ ]:
_source_path = DOCUMENTS_DIR / "regulation-yv5UwjfEILZBCtSQvOHdA.txt"
_raw_text = _source_path.read_text(encoding="utf-8")

BASE_UNITS = split_into_units(_raw_text)[:15]
BASE_TEXT = " ".join(BASE_UNITS)

print(f"Loaded {len(BASE_UNITS)} real sentence units from {_source_path.name}")
print(f"Sample unit: {BASE_UNITS[5]!r}")


## 12. Test 1 — Added Content

In [ ]:
NEW_SENTENCE = "This is a newly added requirement, effective immediately for all operators."
added_variant_text = BASE_TEXT + " " + NEW_SENTENCE

diff_added, result_added = compare_documents(BASE_TEXT, added_variant_text, config)
print(diff_added)
print(result_added.model_dump_json(indent=2))

assert diff_added.added == [NEW_SENTENCE]
assert diff_added.removed == [] and diff_added.modified == []
assert result_added.total_changes == 1
print("Test 1 passed: added sentence correctly detected, nothing else flagged.")


## 13. Test 2 — Removed Content

In [ ]:
removed_unit = BASE_UNITS[5]
removed_variant_units = BASE_UNITS[:5] + BASE_UNITS[6:]
removed_variant_text = " ".join(removed_variant_units)

diff_removed, result_removed = compare_documents(BASE_TEXT, removed_variant_text, config)
print(diff_removed)

assert diff_removed.removed == [removed_unit]
assert diff_removed.added == [] and diff_removed.modified == []
assert result_removed.total_changes == 1
print("Test 2 passed: removed sentence correctly detected, nothing else flagged.")


## 14. Test 3 — Modified Content

In [ ]:
original_unit = BASE_UNITS[5]
reworded_unit = original_unit.replace("Section", "Section (Amended)")
if reworded_unit == original_unit:
    reworded_unit = original_unit + " [reworded for this test]"

modified_variant_units = list(BASE_UNITS)
modified_variant_units[5] = reworded_unit
modified_variant_text = " ".join(modified_variant_units)

diff_modified, result_modified = compare_documents(BASE_TEXT, modified_variant_text, config)
print(diff_modified)

assert diff_modified.modified == [(original_unit, reworded_unit)]
assert diff_modified.added == [] and diff_modified.removed == []
assert result_modified.total_changes == 1
print("Test 3 passed: modified sentence correctly detected as a single replace, not add+remove.")


## 15. Test 4 — Formatting-Only Changes (must NOT be reported)

Reconstructs `BASE_TEXT` with irregular line breaks and doubled spacing,
mimicking real PDF line-wrap artifacts -- no wording changes at all. This is
the core proof that whitespace normalization (Section 6) keeps formatting
noise from ever being reported as a regulatory change.


In [ ]:
words = BASE_TEXT.split(" ")
formatted_parts = []
for i, w in enumerate(words):
    formatted_parts.append(w)
    formatted_parts.append("\n  " if (i + 1) % 7 == 0 else "  ")
formatting_variant_text = "".join(formatted_parts)

assert formatting_variant_text != BASE_TEXT, "Fixture should differ in raw formatting."
assert normalize_for_diff(formatting_variant_text) == normalize_for_diff(BASE_TEXT), \
    "Fixture should be textually identical once whitespace is normalized."

diff_formatting, result_formatting = compare_documents(BASE_TEXT, formatting_variant_text, config)
print(diff_formatting)
print(result_formatting.model_dump_json(indent=2))

assert diff_formatting.added == [] and diff_formatting.removed == [] and diff_formatting.modified == []
assert result_formatting.total_changes == 0
assert result_formatting.overall_change_level == "NONE"
print("Test 4 passed: formatting-only differences produced ZERO reported changes.")


## 16. Test 5 — Previous-Version Identification

Two cases: (a) against the real Phase 3 database, where no genuine revision
pair exists, `find_previous_version()` must correctly return `None` rather
than a false match; (b) against a seeded in-memory database with a real
revision pair, it must correctly find the older one.


In [ ]:
real_conn = get_connection()

no_match = find_previous_version(real_conn, "regulation-yv5UwjfEILZBCtSQvOHdA")
assert no_match is None
print("Test 5a passed: correctly found no previous version in real data (none exists).")

_PUBLICATIONS_SCHEMA = """
CREATE TEMP TABLE publications (
    publication_id   TEXT PRIMARY KEY,
    title             TEXT NOT NULL,
    category          TEXT NOT NULL,
    published_date    TEXT,
    publication_url   TEXT,
    pdf_url           TEXT,
    content_hash      TEXT NOT NULL,
    document_hash     TEXT,
    first_seen_at     TEXT NOT NULL,
    processed_at      TEXT,
    email_sent        INTEGER NOT NULL DEFAULT 0,
    email_sent_at     TEXT,
    status            TEXT NOT NULL DEFAULT \'new\'
);
"""

test_conn = get_connection()
test_conn.execute(_PUBLICATIONS_SCHEMA)
test_conn.cursor().executemany(
    "INSERT INTO publications (publication_id, title, category, content_hash, first_seen_at) VALUES (%s, %s, %s, %s, %s)",
    [
        ("regulation-car-x-rev1", "CAR Section X - Revision 1", "Regulation", "hash1", "2026-01-01T00:00:00+00:00"),
        ("regulation-car-x-rev2", "CAR Section X - Revision 2", "Regulation", "hash2", "2026-06-01T00:00:00+00:00"),
        ("regulation-unrelated", "An Entirely Different Regulation", "Regulation", "hash3", "2026-03-01T00:00:00+00:00"),
    ],
)
test_conn.commit()

found = find_previous_version(test_conn, "regulation-car-x-rev2")
assert found is not None
assert found["publication_id"] == "regulation-car-x-rev1"
matched_title = found["title"]
print(f"Test 5b passed: correctly matched \'{matched_title}\' as the previous version.")

test_conn.close()


## 17. End-to-End Run — a realistic multi-change revision

Combines an addition, a removal, and a modification in one NEW version (a
more realistic simulation of an actual DGCA revision than any single test
above), runs the full pipeline, and saves the result.


In [ ]:
combined_units = list(BASE_UNITS)
combined_units[3] = combined_units[3] + " [operational threshold revised]"
del combined_units[8]
combined_units.append("Operators must submit updated compliance documentation within 30 days of this revision.")
combined_new_text = " ".join(combined_units)

final_diff, final_result = compare_documents(BASE_TEXT, combined_new_text, config)

print(final_result.model_dump_json(indent=2))

output_path = COMPARISONS_DIR / "test-old_vs_test-new.json"
output_path.write_text(final_result.model_dump_json(indent=2), encoding="utf-8")
print(f"Saved comparison result to: {output_path}")


## 18. Results

In [ ]:
print("=" * 70)
print("COMPARISON PIPELINE TEST RESULTS")
print("=" * 70)
print(f"Test 1 (added content)              : PASSED")
print(f"Test 2 (removed content)             : PASSED")
print(f"Test 3 (modified content)            : PASSED")
print(f"Test 4 (formatting-only, zero noise) : PASSED")
print(f"Test 5 (previous-version matching)   : PASSED")
print(f"End-to-end multi-change run          : overall_change_level={final_result.overall_change_level}, "
      f"total_changes={final_result.total_changes}")
print(f"AI comparison mode                   : {'OpenAI (' + config.llm_model + ')' if config.openai_configured() else 'deterministic fallback (no OPENAI_API_KEY)'}")
print("=" * 70)

real_conn.close()


## 19. Next Steps

**Phase 5 status: diff engine, previous-version matching, and AI comparison
(with tested deterministic fallback) built and verified** against real DGCA
text plus precisely constructed added/removed/modified/formatting-only
variants.

### Known limitation

- The AI comparison path (real OpenAI call) is implemented and documented
  but not exercised for real in this environment -- no `OPENAI_API_KEY` is
  configured, same situation as Phase 1's AI summary. All tests here
  validate the deterministic fallback, which produces the identical schema.

### Not started yet (by design)

- Phase 6: AI executive summary generation from a `ComparisonResult`
- Phase 7: Agentic workflow orchestration
- Phase 8: Streamlit dashboard

Waiting for confirmation before starting Phase 6.
